# 🏥 RIVA Unified Dashboard - Readmission + LOS
## نظام موحد للتنبؤ بمخاطر إعادة الدخول ومدة الإقامة
---

In [ ]:
# ================== استيراد المكتبات ==================
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
import joblib
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ المكتبات جاهزة")

In [ ]:
# ================== تحميل النماذج ==================
base = '/content/drive2/MyDrive/RIVA-Maternal'

# Readmission model
read_model = joblib.load(f'{base}/readmission/models/xgb_readmission.pkl')
with open(f'{base}/readmission/models/feature_names.json', 'r') as f:
    read_features = json.load(f)['features']

# LOS model
los_model = joblib.load(f'{base}/los/models/xgb_los_improved.pkl')
with open(f'{base}/los/models/feature_names_improved.json', 'r') as f:
    los_features = json.load(f)['features']

print("✅ النماذج محملة بنجاح")

In [ ]:
# ================== إنشاء الواجهة التفاعلية ==================

# Age slider
age = widgets.IntSlider(value=50, min=18, max=100, step=1, description='العمر:')

# Gender dropdown
gender = widgets.Dropdown(
    options=[('ذكر', 'M'), ('أنثى', 'F')],
    value='M',
    description='الجنس:'
)

# Admission type
admit = widgets.Dropdown(
    options=['EMERGENCY', 'ELECTIVE', 'URGENT'],
    value='EMERGENCY',
    description='نوع الدخول:'
)

# Clinical features
diag = widgets.IntSlider(value=4, min=0, max=20, step=1, description='التشخيصات:')
proc = widgets.IntSlider(value=2, min=0, max=10, step=1, description='الإجراءات:')
med = widgets.IntSlider(value=6, min=0, max=30, step=1, description='الأدوية:')
charlson = widgets.IntSlider(value=2, min=0, max=10, step=1, description='Charlson:')
visits = widgets.IntSlider(value=3, min=0, max=20, step=1, description='الزيارات:')

# Button
predict_btn = widgets.Button(
    description='🔮 توقع',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='50px')
)

# Output
output = widgets.Output()

display(widgets.VBox([age, gender, admit, diag, proc, med, charlson, visits, predict_btn, output]))

In [ ]:
# ================== دالة التوقع ==================
def predict_patient(age_val, gender_val, admit_val, diag_val, proc_val, med_val, charlson_val, visits_val):
    
    patient_data = {
        'age': age_val,
        f'gender_{gender_val}': 1,
        f'admit_{admit_val}': 1,
        'num_diagnoses': diag_val,
        'num_procedures': proc_val,
        'num_medications': med_val,
        'charlson_index': charlson_val,
        'total_visits': visits_val,
        'length_of_stay': 0
    }
    
    # Readmission
    read_input = [patient_data.get(f, 0) for f in read_features]
    X_read = np.array(read_input).reshape(1, -1)
    read_prob = read_model.predict_proba(X_read)[0][1]
    read_pred = 1 if read_prob > 0.5 else 0
    
    # LOS
    los_input = [patient_data.get(f, 0) for f in los_features]
    X_los = np.array(los_input).reshape(1, -1)
    los_pred = los_model.predict(X_los)[0]
    
    return {
        'readmission_prob': read_prob,
        'readmission_risk': 'مرتفع' if read_pred == 1 else 'منخفض',
        'los_pred': round(los_pred, 1)
    }

def on_predict_clicked(b):
    with output:
        clear_output(wait=True)
        
        result = predict_patient(
            age.value,
            gender.value,
            admit.value,
            diag.value,
            proc.value,
            med.value,
            charlson.value,
            visits.value
        )
        
        print("\n" + "="*50)
        print("🏥 نتيجة التوقع الشامل")
        print("="*50)
        print(f"\n🔄 خطر إعادة الدخول: {result['readmission_risk']}")
        print(f"   الاحتمال: {result['readmission_prob']:.1%}")
        print(f"\n⏱️ مدة الإقامة المتوقعة: {result['los_pred']} يوم")
        print("="*50)

predict_btn.on_click(on_predict_clicked)